In [2]:
import os
import json
import pandas as pd
import numpy as np
import cv2
from scipy.signal.windows import gaussian as scipy_gaussian

import matplotlib.pyplot as plt

In [3]:
# 1. Paramètres
CSV_PATH    = "label_annotated.csv"   # votre export CSV de VIA
IMG_DIR     = "dataset_project_iapr2025/train"                # dossier de vos .JPG
OUT_DIR     = "density_maps/"                                 # dossier où sauver les .npz
SIGMA       = 150                                             # écart-type de la gaussienne
CLASS_NAMES = [
    "Jelly_White","Jelly_Milk","Jelly_Black","Amandina","Crème_brûlée",
    "Triangolo","Tentation_noir","Comtesse","Noblesse","Noir_authentique",
    "Passion_au_lait","Arabia","Stracciatella"
]

In [16]:
# 2. Mapping nom → indice
class2idx = {name: i for i, name in enumerate(CLASS_NAMES)}

In [17]:
# 3. Précompute the Gaussian kernel
kern_size = int(6 * SIGMA) | 1   # kernel size, ensure odd
g1d = scipy_gaussian(kern_size, SIGMA)
kernel2d = np.outer(g1d, g1d)
kernel2d /= kernel2d.sum()       # normalize sum to 1
half_k = kern_size // 2

In [18]:
# 4. Charger le CSV
df = pd.read_csv(CSV_PATH)

In [19]:
# 5. Créer le dossier de sortie
os.makedirs(OUT_DIR, exist_ok=True)

In [20]:
# 6. Boucle sur chaque image
for fname, group in df.groupby("filename"):
    # 6.1 Lire l'image pour connaître ses dimensions
    img_path = os.path.join(IMG_DIR, fname)
    img = cv2.imread(img_path)
    if img is None:
        print(f"[ERROR] Impossible de lire l'image : {img_path}")
        continue
    H, W = img.shape[:2]
    
    # 6.2 Initialiser le tenseur de density maps
    density = np.zeros((H, W, len(CLASS_NAMES)), dtype=np.float32)
    
    # 6.3 Pour chaque point annoté, scatter-add le kernel
    for _, row in group.iterrows():
        shape_attr = json.loads(row["region_shape_attributes"])
        x = int(shape_attr["cx"])
        y = int(shape_attr["cy"])
        choco_type = json.loads(row["region_attributes"])["Choco_Type"]
        c = class2idx.get(choco_type, None)
        if c is None:
            continue
        
        # bounds for scatter-add
        y0 = max(0,   y - half_k)
        y1 = min(H,   y + half_k + 1)
        x0 = max(0,   x - half_k)
        x1 = min(W,   x + half_k + 1)
        ky0 = half_k - (y - y0)
        ky1 = half_k + (y1 - y)
        kx0 = half_k - (x - x0)
        kx1 = half_k + (x1 - x)
        
        density[y0:y1, x0:x1, c] += kernel2d[ky0:ky1, kx0:kx1]
    
    # 6.4 Conversion en float16 pour réduire la taille
    density = density.astype(np.float16)
    
    # 6.5 Sauvegarder au format compressé .npz
    out_path = os.path.join(OUT_DIR, fname.replace(".JPG", ".npz"))
    np.savez_compressed(out_path, density=density)
    
    print(f"Saved compressed density map for {fname} → {out_path}")


Saved compressed density map for L1000756.JPG → density_maps/L1000756.npz
Saved compressed density map for L1000763.JPG → density_maps/L1000763.npz
Saved compressed density map for L1000765.JPG → density_maps/L1000765.npz


KeyboardInterrupt: 

In [1]:
# exemple de chargement d'une density map et de l'image correspondante
npz = np.load("density_maps/L1000763.npz")
density = npz["density"]    # shape (H, W, n_classes)
img = cv2.imread("dataset_project_iapr2025/train/L1000763.JPG")[:,:,::-1]

# visualisation de la somme sur les canaux (densité totale)
total_density = density.sum(axis=-1)

fig, axes = plt.subplots(1,2, figsize=(12,5))
axes[0].imshow(img); axes[0].set_title("Image originale"); axes[0].axis("off")
axes[1].imshow(total_density, cmap="jet"); axes[1].set_title("Density map (somme)"); axes[1].axis("off")
plt.show()

# vérifiez aussi la somme pour estimer le nombre d'objets
print("Nombre estimé d'objets sur l'image :", total_density.sum())


NameError: name 'np' is not defined

# Neural network :

In [4]:
import os
import json
import random
from glob import glob

import cv2
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
from tqdm import tqdm

In [5]:
# ─── 1. CONFIG ────────────────────────────────────────────────────────────────
DM_DIR      = "density_maps"
N_CLASSES = len(CLASS_NAMES)
DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH     = 4
EPOCHS    = 30
LR        = 1e-4
SEED      = 42

# Réglage du seed
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

In [33]:
class ChocolateDataset(Dataset):
    def __init__(self, files, img_dir, dm_dir, transform=None):
        self.files     = files
        self.img_dir   = img_dir
        self.dm_dir    = dm_dir
        self.transform = transform
        # taille cible
        self.target_h, self.target_w = 512, 512

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname   = self.files[idx]
        img_path = os.path.join(self.img_dir, fname)
        dm_path  = os.path.join(self.dm_dir, fname.replace(".JPG", ".npz"))

        # --- 1) Lecture numpy de l'image et de la density-map ---
        img_np = cv2.imread(img_path)
        if img_np is None:
            raise RuntimeError(f"[Dataset] Impossible de lire l'image : {img_path}")
        img_np = img_np[:, :, ::-1].astype(np.float32) / 255.0  # BGR→RGB, [0,1]

        dm_np = np.load(dm_path)["density"]   # dtype=float16
        dm_np = dm_np.astype(np.float32)      # on garde float32 pour la loss
        # dm_np shape = (H, W, C)

        # --- 2) Resize numpy avant conversion ---
        # images : interpolation bilinéaire
        img_np = cv2.resize(
            img_np,
            (self.target_w, self.target_h),
            interpolation=cv2.INTER_AREA,
        )
        # density : on veut conserver la somme spatiale = count
        # on resize canal par canal puis on rééchelle
        H0, W0 = dm_np.shape[:2]
        scale_factor = (H0 * W0) / (self.target_h * self.target_w)
        # tranpose pour CV resize : (C, H, W)
        dm_chw = dm_np.transpose(2, 0, 1)
        dm_resized = np.zeros((dm_chw.shape[0], self.target_h, self.target_w), dtype=np.float32)
        for c in range(dm_chw.shape[0]):
            patch = cv2.resize(
                dm_chw[c],
                (self.target_w, self.target_h),
                interpolation=cv2.INTER_AREA,
            )
            dm_resized[c] = patch * scale_factor

        # --- 3) Conversion en Tensors (CHW) ---
        # image → float32 Tensor (3, H, W)
        img_tensor = torch.from_numpy(img_np.transpose(2, 0, 1)).float()
        # density → (C, H, W)
        dm_tensor  = torch.from_numpy(dm_resized).float()

        # --- 4) éventuel transform sur le Tensor ---
        if self.transform:
            img_tensor = self.transform(img_tensor)

        return img_tensor, dm_tensor


# Lire toutes les images
all_files = [os.path.basename(p) for p in glob(os.path.join(IMG_DIR, "*.JPG"))]
train_files, test_files = train_test_split(all_files, test_size=0.2, random_state=SEED)
train_files, val_files  = train_test_split(train_files, test_size=0.1, random_state=SEED)

# DataLoaders
train_ds = ChocolateDataset(train_files, IMG_DIR, DM_DIR)
val_ds   = ChocolateDataset(val_files,   IMG_DIR, DM_DIR)
test_ds  = ChocolateDataset(test_files,  IMG_DIR, DM_DIR)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=True)


In [34]:
# ─── 3. UNet SIMPLE ─────────────────────────────────────────────────────────────
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)

class UNet(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.down1 = DoubleConv(3,  64)
        self.pool  = nn.MaxPool2d(2)
        self.down2 = DoubleConv(64, 128)
        self.down3 = DoubleConv(128,256)
        self.down4 = DoubleConv(256,512)

        self.up3 = nn.ConvTranspose2d(512,256,2,stride=2)
        self.conv3 = DoubleConv(512,256)
        self.up2 = nn.ConvTranspose2d(256,128,2,stride=2)
        self.conv2 = DoubleConv(256,128)
        self.up1 = nn.ConvTranspose2d(128,64,2,stride=2)
        self.conv1 = DoubleConv(128,64)

        self.final = nn.Conv2d(64, n_classes, kernel_size=1)

    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(self.pool(d1))
        d3 = self.down3(self.pool(d2))
        d4 = self.down4(self.pool(d3))

        u3 = self.up3(d4)
        u3 = torch.cat([u3, d3], dim=1)
        u3 = self.conv3(u3)

        u2 = self.up2(u3)
        u2 = torch.cat([u2, d2], dim=1)
        u2 = self.conv2(u2)

        u1 = self.up1(u2)
        u1 = torch.cat([u1, d1], dim=1)
        u1 = self.conv1(u1)

        return self.final(u1)

# Instanciation
model = UNet(N_CLASSES).to(DEVICE)

In [29]:
from torchsummary import summary
summary(model, (3, 512, 512), batch_size=1)

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1          [1, 64, 512, 512]           1,728
       BatchNorm2d-2          [1, 64, 512, 512]             128
              ReLU-3          [1, 64, 512, 512]               0
            Conv2d-4          [1, 64, 512, 512]          36,864
       BatchNorm2d-5          [1, 64, 512, 512]             128
              ReLU-6          [1, 64, 512, 512]               0
        DoubleConv-7          [1, 64, 512, 512]               0
         MaxPool2d-8          [1, 64, 256, 256]               0
            Conv2d-9         [1, 128, 256, 256]          73,728
      BatchNorm2d-10         [1, 128, 256, 256]             256
             ReLU-11         [1, 128, 256, 256]               0
           Conv2d-12         [1, 128, 256, 256]         147,456
      BatchNorm2d-13         [1, 128, 256, 256]             256
             ReLU-14         [1, 128, 2

In [35]:
# ─── 4. LOSS, OPTI, SCHEDULER ───────────────────────────────────────────────────
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,
                                                       mode='min',
                                                       factor=0.5,
                                                       patience=3,
                                                       verbose=True)

In [36]:
# ─── 5. FONCTIONS D’ENTRAÎNEMENT & ÉVAL ─────────────────────────────────────────
def train_epoch(loader, model, criterion, optimizer):
    model.train()
    running_loss = 0.0
    for imgs, dms in tqdm(loader, desc="Train", leave=False):
        imgs = imgs.to(DEVICE)
        dms  = dms.to(DEVICE)
        preds = model(imgs)
        loss  = criterion(preds, dms)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
    return running_loss / len(loader.dataset)

def eval_epoch(loader, model, criterion):
    model.eval()
    running_loss = 0.0
    mae_per_class = torch.zeros(N_CLASSES, device=DEVICE)
    with torch.no_grad():
        for imgs, dms in tqdm(loader, desc="Eval", leave=False):
            imgs = imgs.to(DEVICE)
            dms  = dms.to(DEVICE)
            preds = model(imgs)
            running_loss += criterion(preds, dms).item() * imgs.size(0)

            # counts
            pred_cnt = preds.sum(dim=(2,3))  # (B, C)
            true_cnt = dms.sum(dim=(2,3))    # (B, C)
            mae_per_class += torch.abs(pred_cnt - true_cnt).sum(dim=0)

    loss = running_loss / len(loader.dataset)
    mae  = (mae_per_class / len(loader.dataset)).cpu().numpy()  # vecteur C
    return loss, mae

In [37]:
# ─── 6. BOUCLE PRINCIPALE ───────────────────────────────────────────────────────
best_val_loss = float('inf')
for epoch in range(1, EPOCHS+1):
    tr_loss = train_epoch(train_loader, model, criterion, optimizer)
    val_loss, val_mae = eval_epoch(val_loader, model, criterion)
    scheduler.step(val_loss)

    print(f"Époch {epoch:02d} | Train Loss: {tr_loss:.4f} "
          f"| Val Loss: {val_loss:.4f} | Mean MAE: {val_mae.mean():.3f}")

    # sauvegarde du meilleur modèle (sur la loss de validation)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_choco_unet.pth")
        print("→ Modèle sauvegardé!")

KeyboardInterrupt: 

In [ ]:
# ─── 7. TEST FINAL ───────────────────────────────────────────────────────────────
model.load_state_dict(torch.load("best_choco_unet.pth"))
test_loss, test_mae = eval_epoch(test_loader, model, criterion)
print(f"\n=== TEST FINAL ===\nTest Loss: {test_loss:.4f} "
      f"| MAE par classe:\n")
for cname, mae_val in zip(CLASS_NAMES, test_mae):
    print(f"  - {cname:20s}: {mae_val:.3f}")
print(f"\nMAE globale : {test_mae.mean():.3f}")